In [ ]:
!pip install -q fastparquet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from itertools import product
import seaborn as sns
import warnings
import requests
import pyarrow
import fastparquet

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.0f}'.format)

In [ ]:
#%reset -f

In [ ]:
# данные о продажах
data  = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/gmv_dataset.parquet', engine='fastparquet')

# данные о праздниках 
holidays = pd.read_excel('/kaggle/input/datasets/faibus/diploma/holidays.xlsx')
holidays['Holidays'] = pd.to_datetime(holidays['Holidays'], format='%Y.%m.%d')

# данные о погоде
weather = pd.read_csv('/kaggle/input/datasets/faibus/diploma/weather.csv')

##### Создание фичи с температурой воздуха 

В рамках работы появилась идея использовать информацию о температуре воздуха в регионе для улучшения прогнозирования, ведь не всегда люди покупают товары в соответствии с календарем, иногда это происходит в связи с определенными температурными изменениями. Я выгрузил температуру воздуха и погоду по основным регионам доставки товаров.


In [ ]:
weather.head(1)

## Описание данных

---

В нашем датасете 10.8 млн строк. Данные содержат информацию о продажах товаров интернет-магазина.

- Выбранные категории — категории, связанные с автомобилями.
- Временной горизонт — с января 2024 по сентябрь 2025

**Описание колонок:**
- YYYYMM — дата в формате ГОДМЕСЯЦ
- Date — дата в формате 'YYYY-MM-DD'
- Category3 — категория 3-го уровня. У нас всего одна категория 3-го уровня.
- Category4 — категория 4-го уровня. Всего таких категорий четыре.
- Brand — бренд товара. Всего в нашем датасете 879 уникальных брендов
- SupplierID — айди поставщика, который поставляет товар. Всего в нашем датасете 2765 уникальных поставщиков
- SellerID — айди магазина, который продаёт товар. Всего в нашем датасете 2960 уникальных магазинов
- SKU_ID — айди товара. Всего в нашем датасете около 91 тыс. уникальных товаров
- is_promo — бинарная переменная. Признак участия товара в промо в данный день (1 - товар участвовал в промо, 0 - не участвовал)
- stock_qty — остатки товара на складах на начало дня (берутся только те, что доступны к продаже).
- GMV_wo_VAT — товарооборот (D-R) в денежных единицах. (D-R) - это (Delivered - Returned), т.е. товарооборот доставленных заказов за вычетом возвратов
- GMV_wo_VAT_Acc — товарооборот (Acc) в денежных единицах. (Acc) - это только принятые заказы
- Items_Acc — объём принятых заказов в штуках

---

##### Ответы на основные вопросы

- В чём разница между SupplierID и SellerID? — Поставщик - это компания, которая поставляет товары на склад. Магазин - лицо, которое продаёт конечному потребителю. Например, компания занимается продажей электроники, продаёт бренды Samsung и Apple. Такая компания может захотеть продавать товары различных брендов от имени различных магазинов - 'Иванов оригинальная техника Apple' и 'Иванов оригинальная техника Samsung'. В таких случаях у товаров будет один и тот же SupplierID, но различные SellerID.
- Может ли одна SKU_ID принадлежать различным магазинам? — Нет, не может. SKU_ID уникальна и принадлежит только одному магазину, поставщику.

### Обработка пропусков

- В нашем датасете есть пропуски в столбце Brand. Всего таких записей 260. Пропуски в брендах составляют всего 0.000% данных и 0.000% оборота. Рекомендуется исключить эти данные из анализа.

In [ ]:
data.isna().sum()

# Смотрим на пропуски в брендах
brand_missing = data[data['Brand'].isna()]

print('Строки без бренда:')
print(f'Всего: {len(brand_missing):,}')
print(f'Это {len(brand_missing)/len(data)*100:.1f}% от всех данных')
print()

print('Считаем уникальные товары и поставщиков:')
print(f'Уникальных товаров без бренда: {brand_missing["SKU_id"].nunique():,}')
print(f'Поставщиков без брендов: {brand_missing["SupplierID"].nunique():,}')
print(f'Магазинов без брендов: {brand_missing["SellerID"].nunique():,}')
print()

print('Оборот по строкам без бренда:')
total_sales = brand_missing['GMV_wo_VAT_Acc'].sum()
total_orders = brand_missing['Items_Acc'].sum()
print(f'Общий оборот: {total_sales:,.0f} руб.')
print(f'Количество заказов: {total_orders:,} шт.')
print()

# Сравниваем с общими цифрами
all_sales = data['GMV_wo_VAT_Acc'].sum()
print(f'Оборот без брендов от общего: {total_sales/all_sales*100:.3f}%')
print()

##### Удаление строк с пропусками

In [ ]:
# Удаляем строки с пропущенными брендами
data = data[data['Brand'].notna()]

data.isna().sum()

### Исследование данных
---
В данных заметны определенные тенденции и сезонности

- GMV устойчиво растёт с января по сентябрь в 24 и 25 годах
- GMV стагнирует с апреля по июнь в каждом из годов и продолжает рост начиная с июля

Сезонные скачки GMV с февраля по апрель:
- MoM март 2024 = 1.29, MoM март 25 = 1.37
- MoM апрель 24 = 1.26, MoM апрель 25 = 1.18

---

In [ ]:
monthly_stats = data \
    .groupby('YYYYMM') \
    .agg({'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 14))

sns.barplot(
    data=monthly_stats,
    x='YYYYMM',
    y='GMV_wo_VAT_Acc',
    color='steelblue',
    edgecolor='black',
    linewidth=0.5,
    ax=ax1
)

# Настройки первого графика
ax1.set_title('Динамика GMV_Acc по месяцам', fontsize=14, pad=20)
ax1.set_xlabel('')
ax1.set_ylabel('GMV_wo_VAT_Acc', fontsize=12)
ax1.set_yticklabels([])

sns.barplot(
    data=monthly_stats,
    x='YYYYMM',
    y='Items_Acc',
    color='salmon',
    edgecolor='black',
    linewidth=0.5,
    ax=ax2
)

ax2.set_title('Динамика Items_Acc по месяцам', fontsize=14, pad=20)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Items_Acc', fontsize=12)
ax2.set_yticklabels([])

plt.xticks()
plt.tight_layout()
plt.show()

In [ ]:
monthly_stats['date'] = pd.to_datetime(monthly_stats['YYYYMM'].astype(str), format='%Y%m')
monthly_stats['month'] = monthly_stats['date'].dt.month
monthly_stats['year'] = monthly_stats['date'].dt.year
monthly_stats.set_index('date', inplace=True)
monthly_stats['GMV_MoM_pct'] = monthly_stats['GMV_wo_VAT_Acc'].pct_change() * 100

monthly_stats.pivot(index = 'month', columns = 'year', values = 'GMV_MoM_pct')

Почти абсолютная корреляция между GMV_wo_VAT_Acc и Items_Acс.

In [ ]:
daily_gmv_items = data \
    .groupby('Date') \
    .agg({'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index()

correlation_matrix = daily_gmv_items[['GMV_wo_VAT_Acc', 'Items_Acc']].corr()
corr_value = correlation_matrix.loc['GMV_wo_VAT_Acc', 'Items_Acc']

print(f"Корреляция между GMV_wo_VAT_Acc и Items_Acc: {corr_value:.3f}")

### Анализ в разрезе категорий 4-го уровня
---
В данных заметен сильный дисбаланс. 80 и более процентов оборота приходится на одну категорию 4-го уровня. Это наталкивает на вопрос об уровне информативности категорий 4-го уровня в качестве регрессоров.

---

In [ ]:
monthly_category4 = data.groupby(['YYYYMM', 'Category4'])['GMV_wo_VAT_Acc'].sum().reset_index()

pivot_category4 = monthly_category4.pivot(
    index='YYYYMM', 
    columns='Category4', 
    values='GMV_wo_VAT_Acc', 
).fillna(0)

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(14, 8))

colors = sns.color_palette("Set2", len(pivot_category4.columns))
pivot_category4.plot(
    kind='bar',
    stacked=True,
    color=colors,
    edgecolor='black',
    linewidth=0.5,
    width=0.8,
    ax=ax, 
    legend = False
)

# Настройка графика
ax.set_title('Динамика GMV_Acc по месяцам (Category4)', fontsize=14, pad=20)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('GMV_wo_VAT_Acc', fontsize=12)
ax.set_yticklabels([])

# Убираем вертикальные линии сетки
ax.grid(True, axis='y', alpha=0.3)
ax.grid(False, axis='x')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Можно предположить корреляцию между категориями 4-го уровня.

In [ ]:
monthly_category4['YYYYMM'] =  monthly_category4['YYYYMM'].astype(str)

plt.figure(figsize=(14, 8))
sns.lineplot(data = monthly_category4, y = 'GMV_wo_VAT_Acc', hue = 'Category4', x = 'YYYYMM', legend = False)
plt.title('Динамика GMV_Acc кат4 по месяцам', fontsize=12)
plt.xticks(rotation=90)
plt.yticks([])
plt.tight_layout()
plt.grid(True, axis='y', alpha=0.3)
plt.grid(True, axis='x')
plt.show()

### Анализ уникальных значений
---

В датасете заметно стабильное увеличение количества уникальных брендов, селлеров и SKU.

- кол-во уникальных селлеров почти совпадает с кол-ом уникальных поставщиков -> рационально использовать что-то одно в качестве экзогенной переменной в моделях
- кол-во уникальных брендов почти в 3 раза меньше кол-ва уникальных продавцов. На сентябрь 25-го наблюдается 555 уникальных брендов
- кол-во уникальных SKU стагнировало с сентября 24-го по июль 25-го, после чего продолжило свой рост. В сентябре 25-го продавалось около 26 тысяч уникальных SKU

---

In [ ]:
unique_values = data \
    .groupby('YYYYMM') \
    .agg({'Brand': 'nunique', 'SupplierID': 'nunique', 'SellerID': 'nunique', 'SKU_id': 'nunique'}) \
    .reset_index()

unique_values = unique_values.rename(columns={
    'Brand': 'Brand_Count',
    'SupplierID': 'SupplierID_Count',
    'SellerID': 'SellerID_Count',
    'SKU_id': 'SKU_id_Count'
})

unique_values

In [ ]:
unique_values = data \
    .groupby('YYYYMM') \
    .agg({'Brand': 'nunique', 'SupplierID': 'nunique', 'SellerID': 'nunique', 'SKU_id': 'nunique'}) \
    .reset_index()

# Преобразуем YYYYMM в datetime (первый день месяца)
unique_values['Date'] = pd.to_datetime(unique_values['YYYYMM'].astype(str), format='%Y%m')

# Преобразуем данные в long format для hue
unique_long = unique_values.melt(
    id_vars=['YYYYMM', 'Date'],
    value_vars=['Brand', 'SupplierID', 'SellerID', 'SKU_id'],
    var_name='Metric',
    value_name='Count'
)

# Разделяем на две группы: малые метрики и SKU
small_metrics = unique_long[unique_long['Metric'] != 'SKU_id']
sku_metric = unique_long[unique_long['Metric'] == 'SKU_id']

# Создаем графики
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
sns.set_style("whitegrid")

# График 1: Brand, SupplierID, SellerID
sns.lineplot(
    data=small_metrics,
    x='Date',
    y='Count',
    hue='Metric',
    marker='o',
    markersize=6,
    linewidth=2,
    ax=ax1
)

ax1.set_title('Уникальные значения', fontsize=14, pad=20)
ax1.set_xlabel('')
ax1.set_ylabel('Количество', fontsize=12)

# Форматируем даты
ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=90)

# Добавляем подписи для каждой линии в последней точке
for metric in ['Brand', 'SupplierID', 'SellerID']:
    last_row = unique_values[['Date', metric]].iloc[-1]
    ax1.text(
        last_row['Date'],
        last_row[metric],
        f'  {last_row[metric]}',
        va='center',
        fontsize=9,
        fontweight='bold'
    )

# График 2: SKU_id
sns.lineplot(
    data=sku_metric,
    x='Date',
    y='Count',
    hue='Metric',
    marker='o',
    markersize=6,
    linewidth=2,
    ax=ax2
)

ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Количество SKU', fontsize=12)

# Форматируем даты
ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

# Подпись для последней точки SKU
last_sku = unique_values[['Date', 'SKU_id']].iloc[-1]
ax2.text(
    last_sku['Date'],
    last_sku['SKU_id'],
    f'  {last_sku["SKU_id"]:,}',
    va='center',
    fontsize=9,
    fontweight='bold'
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### Анализ средней стоимости товаров
---

- средняя стоимость товаров в наблюдаемых категориях стагнирует
- некоторые категории демонстрируют высокую корреляцию в ценах
- только одна из четырех категорий имеет значимые различия в динамике и характере движения средней стоимости товара

---

In [ ]:
AIV_data = data \
    .groupby(['YYYYMM', 'Category4']) \
    .agg({'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index()

AIV_data['YYYYMM'] = pd.to_datetime(AIV_data['YYYYMM'].astype(str), format='%Y%m')
AIV_data['AIV'] = AIV_data['GMV_wo_VAT_Acc'] / AIV_data['Items_Acc']

plt.figure(figsize=(14, 8))
sns.lineplot(data = AIV_data, x = 'YYYYMM', y = 'AIV', hue = 'Category4', legend = False)
plt.title('Динамика средней стоимости товаров кат4 по месяцам', fontsize=12)
plt.xlabel('Month', fontsize=12)
plt.ylabel('AIV', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True, axis='y', alpha=0.3)
plt.grid(True, axis='x')
plt.show()

In [ ]:
def create_gmv_datasets(data, columns):
    """
    Создает динамические датасеты для указанных колонок
    
    Args:
        data: исходный DataFrame
        columns: список колонок для анализа
    
    Returns:
        dict: словарь с датасетами {название_колонки: DataFrame}
    """
    total_gmv = data['GMV_wo_VAT_Acc'].sum()
    datasets = {}
    
    for column in columns:
        gmv_df = data.groupby(column)['GMV_wo_VAT_Acc'].sum().reset_index()
        gmv_df = gmv_df.sort_values('GMV_wo_VAT_Acc', ascending=False).reset_index(drop=True)
        gmv_df['Share_%'] = (gmv_df['GMV_wo_VAT_Acc'] / total_gmv) * 100
        gmv_df['Cumulative_Share_%'] = gmv_df['Share_%'].cumsum()
        
        # Сохраняем в словарь с динамическим именем
        dataset_name = f"{column}_gmv"
        datasets[dataset_name] = gmv_df
    
    return datasets

# Использование
columns_to_analyze = ['Brand', 'SellerID', 'SupplierID', 'SKU_id']
gmv_datasets = create_gmv_datasets(data, columns_to_analyze)

# Теперь можно обращаться к датасетам по имени
print("Доступные датасеты:")
print(list(gmv_datasets.keys()))

Заметна высокая концентрация оборота вокруг определенных брендов и селлеров. На один бренд может приходиться до 16% оборота. На одного селлера может приходиться до 15% всего оборота.

Есть вероятность, что информация о том, к какому бренду / селлеру принадлежит товар может иметь прогностическую силу.

In [ ]:
sns.set_style("whitegrid")
plt.figure(figsize=(12, 8))

# Берем топ-20 брендов
top_n = 20
top_brands = gmv_datasets['Brand_gmv'].head(top_n).copy()

# Создаем градиент цветов
colors = plt.cm.tab20(np.linspace(0, 1, top_n))

# Строим горизонтальный барплот
bars = plt.barh(range(top_n), top_brands['GMV_wo_VAT_Acc'], 
                color=colors, edgecolor='black', linewidth=0.5)

# Устанавливаем названия брендов по оси Y
plt.yticks(range(top_n), top_brands['Brand'], fontsize=10)
plt.gca().invert_yaxis()

plt.title(f'Топ-{top_n} брендов по обороту (GMV)', fontsize=14, pad=20)
plt.xlabel('GMV_wo_VAT_Acc', fontsize=10)
plt.ylabel('Бренд', fontsize=10)
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
sns.set_style("whitegrid")
plt.figure(figsize=(12, 8))

# Берем топ-20 брендов
top_n = 20
top_brands = gmv_datasets['SellerID_gmv'].head(top_n).copy()

# Создаем градиент цветов
colors = plt.cm.tab20(np.linspace(0, 1, top_n))

# Строим горизонтальный барплот
bars = plt.barh(range(top_n), top_brands['GMV_wo_VAT_Acc'], 
                color=colors, edgecolor='black', linewidth=0.5)

# Устанавливаем названия брендов по оси Y
plt.yticks(range(top_n), top_brands['SellerID'], fontsize=10)
plt.gca().invert_yaxis()

plt.title(f'Топ-{top_n} селлеров по обороту (GMV)', fontsize=14, pad=20)
plt.xlabel('GMV_wo_VAT_Acc', fontsize=10)
plt.ylabel('Селлер', fontsize=10)
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()


**Исследование влияния дня недели на продажи**

---
- продажи в выходные в среднем ниже, чем продажи в будние дни
- суббота самый низкий день по продажам
- тенденция на более низкие продажи в выходные дни сохраняется на уровне топ-100 SKU
---

In [ ]:
week_day_table = data \
    .groupby('dayofweek') \
    .agg({'Date': 'nunique', 'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index() \
    .rename(columns = {'Date':'number_of_days', 'GMV_wo_VAT_Acc': 'GMV', 'Items_Acc':'Items'})

week_day_table['avg_GMV'] = week_day_table['GMV'] / week_day_table['number_of_days']
week_day_table['avg_Items'] = week_day_table['Items'] / week_day_table['number_of_days']

display_table = week_day_table.copy()
columns_to_hide = ['GMV', 'Items', 'avg_GMV', 'avg_Items']

for col in columns_to_hide:
    display_table[col] = '***'

display_table

In [ ]:
data['dayofweek'] = data['Date'].dt.dayofweek + 1

week_day_table = data \
    .groupby('dayofweek') \
    .agg({'Date': 'nunique', 'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index() \
    .rename(columns = {'Date':'number_of_days', 'GMV_wo_VAT_Acc': 'GMV', 'Items_Acc':'Items'})

week_day_table['avg_GMV'] = week_day_table['GMV'] / week_day_table['number_of_days']
week_day_table['avg_Items'] = week_day_table['Items'] / week_day_table['number_of_days']

week_day_dates_table = data \
    .groupby(['dayofweek', 'Date']) \
    .agg({'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index() \
    .rename(columns = {'GMV_wo_VAT_Acc': 'GMV', 'Items_Acc':'Items'})

plt.figure(figsize=(15, 15))

# Boxplot для GMV
plt.subplot(2, 1, 1)
sns.boxplot(data=week_day_dates_table, x='dayofweek', y='GMV')
plt.title('Распределение GMV по дням недели')
plt.xlabel('День недели (1=Пн, 7=Вс)')
plt.ylabel('GMV')
plt.yticks([])

# Boxplot для Items
plt.subplot(2, 1, 2)
sns.boxplot(data=week_day_dates_table, x='dayofweek', y='Items')
plt.title('Распределение QTY по дням недели')
plt.xlabel('День недели (1=Пн, 7=Вс)')
plt.ylabel('QTY')
plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
sku_table = gmv_datasets['SKU_id_gmv']

top_100_sku = list(sku_table.head(100)['SKU_id'])

week_day_dates_table = data \
    .query(" SKU_id in @top_100_sku ") \
    .groupby(['dayofweek', 'Date']) \
    .agg({'GMV_wo_VAT_Acc': 'sum', 'Items_Acc': 'sum'}) \
    .reset_index() \
    .rename(columns = {'GMV_wo_VAT_Acc': 'GMV', 'Items_Acc':'Items'})

plt.figure(figsize=(15, 15))

# Boxplot для GMV
plt.subplot(2, 1, 1)
sns.boxplot(data=week_day_dates_table, x='dayofweek', y='GMV')
plt.title('Распределение GMV по дням недели для топ-100 SKU')
plt.xlabel('День недели (1=Пн, 7=Вс)')
plt.ylabel('GMV')
plt.yticks([])

# Boxplot для Items
plt.subplot(2, 1, 2)
sns.boxplot(data=week_day_dates_table, x='dayofweek', y='Items')
plt.title('Распределение QTY по дням недели для топ-100 SKU')
plt.xlabel('День недели (1=Пн, 7=Вс)')
plt.ylabel('QTY')
plt.yticks([])

plt.tight_layout()
plt.show()